##### Google Earth Engine Account is required (https://earthengine.google.com/signup/) and installing GEEMAP (https://geemap.org/installation/)

In [4]:
ee.Authenticate()

True

In [3]:
import ee, geemap, math
ee.Initialize()

##### C-factor, one of the important factors for RUSLE annual soil erosion estimation, was calculated here, adapting the approach from Van der Knijff et al. (2000). Non-soil surfaces (built-up/impervious surface and permanent water body) were also masked out and assigned a value of zero so they do not inflate the C outcome. Permanent water bodies were masked using the Sentinel-2 SCL band and the ESA WorldCover ('ESA/WorldCover/v200/2021') (https://doi.org/10.5281/zenodo.7254221). Built-up/impervious surfaces were masked using the ESA WorldCover and Dynamic World ('GOOGLE/DYNAMICWORLD/V1') (https://doi.org/10.1038/s41597-022-01307-4)

##### water and built-up are assigned C=0

In [1]:
# ============================================================
# RUSLE C FACTOR from Sentinel-2 using NDVI (robust bands)
# Water and built-up/impervious set to numeric zero (not masked)
# ============================================================

import ee, geemap, math
ee.Initialize()

# ---------- USER INPUTS ----------
aoi_fc = geemap.shp_to_ee("output/ssurgo_soil_polygon.shp")
aoi = aoi_fc.geometry().dissolve(ee.ErrorMargin(30))

START_DATE = '2024-01-01'
END_DATE   = '2025-01-01'
SCALE_M    = 10

# NDVI→C parameters
ALPHA = 2
BETA  = 1

# ---------- Sentinel-2 collection ----------
s2_raw = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi)
          .filterDate(START_DATE, END_DATE))
# .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))  # optional speedup

def mask_s2_sr(img):
    """
    Mask clouds (QA60 bits 10/11) + bad SCL classes, but DO NOT mask water here.
    Then select red/nir robustly (B04/B4 and B08/B8).
    """
    qa60 = img.select('QA60')
    cloud = qa60.bitwiseAnd(1 << 10).neq(0).Or(qa60.bitwiseAnd(1 << 11).neq(0))

    bands = img.bandNames()
    scl_present = bands.contains('SCL')
    scl = ee.Image(ee.Algorithms.If(scl_present, img.select('SCL'), ee.Image(0)))

    # Bad SCL classes: 1=saturated/defective, 3=cloud shadow, 8=cloud, 9=high cloud, 10=cirrus
    # (intentionally keep SCL==6 water unmasked; we'll set C=0 later)
    scl_bad = scl.eq(1).Or(scl.eq(3)).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10))
    keep = cloud.Not().And(ee.Image(ee.Algorithms.If(scl_present, scl_bad.Not(), ee.Image(1))))

    red_name = ee.String(ee.Algorithms.If(bands.contains('B04'), 'B04', 'B4'))
    nir_name = ee.String(ee.Algorithms.If(bands.contains('B08'), 'B08', 'B8'))
    return img.updateMask(keep).select(ee.List([red_name, nir_name]), ['red','nir'])

s2 = s2_raw.map(mask_s2_sr)
s2_med = s2.median().clip(aoi)

# ---------- Water mask (SCL frequency + WorldCover permanent water) ----------
def water_from_scl(img):
    scl = ee.Image(ee.Algorithms.If(img.bandNames().contains('SCL'), img.select('SCL'), ee.Image(0)))
    return scl.eq(6)  # 6 = water

# Fraction of observations flagged water across the date range
water_frac = s2_raw.map(water_from_scl).mean().clip(aoi)
water_scl  = water_frac.gt(0.5)  # tune threshold: 0.3–0.7 for seasonal water

# Permanent water from WorldCover (class 80)
worldcover = ee.Image('ESA/WorldCover/v200/2021').select('Map').clip(aoi)
water_wc   = worldcover.eq(80)

water_mask = water_scl.Or(water_wc)

# ---------- Built-up / Impervious (WorldCover + Dynamic World) ----------
# WorldCover built-up (class 50)
built_wc = worldcover.eq(50)

# Dynamic World built probability (for recency within your START/END dates)
dw = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
      .filterBounds(aoi)
      .filterDate(START_DATE, END_DATE))
built_dw = dw.select('built').median().gt(0.6)  # adjust 0.5–0.7 for your area

# Combine and slightly dilate to catch narrow roads/edges
built_mask = built_wc.Or(built_dw).focal_max(radius=1, units='pixels')

# ---------- Non-soil mask (impervious OR water) ----------
nonsoil_mask = built_mask.Or(water_mask)
land_mask = nonsoil_mask.Not()

# ---------- NDVI ----------
red = s2_med.select('red').toFloat()
nir = s2_med.select('nir').toFloat()
ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

# ---------- C from NDVI ----------
# C = exp(-ALPHA * max(NDVI,0) / (BETA - NDVI)), clamped to [0,1]; NDVI<=0 → C=1
eps = 1e-6
ndvi_pos = ndvi.max(0)
den = ee.Image.constant(BETA).subtract(ndvi.add(eps))  # β - (NDVI+eps)
ratio = ndvi_pos.divide(den)
C_base = ratio.multiply(-ALPHA).exp().where(ndvi.lte(0), 1).clamp(0, 1)

# ---- Force non-soil (water OR built) to numeric zero (not masked) ----
C = C_base.where(nonsoil_mask, 0).rename('C_from_NDVI')

# ---------- Diagnostics ----------
# Mean C over land only (excludes non-soil in the mean)
meanC_land = C.updateMask(land_mask).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi,
    scale=SCALE_M,
    bestEffort=True,
    maxPixels=1e9,
    tileScale=4
).get('C_from_NDVI')

# Percent area that is non-soil
px_area = ee.Image.pixelArea()
aoi_area_m2 = px_area.reduceRegion(ee.Reducer.sum(), aoi, SCALE_M, bestEffort=True, maxPixels=1e9, tileScale=4).get('area')
nonsoil_area_m2 = px_area.updateMask(nonsoil_mask).reduceRegion(ee.Reducer.sum(), aoi, SCALE_M, bestEffort=True, maxPixels=1e9, tileScale=4).get('area')
print('Mean C (land only):', meanC_land.getInfo())
print('AOI area m^2:', aoi_area_m2.getInfo())
print('Non-soil (water or built) area m^2:', nonsoil_area_m2.getInfo())

# ---------- Map ----------
m = geemap.Map()
m.addLayer(s2_med, {'bands':['nir','red','red'], 'min':0, 'max':3000}, 'S2 composite')
m.addLayer(ndvi,   {'min':-0.2, 'max':1, 'palette':['#fdfbd3','#7fc97f','#1b7837']}, 'NDVI')

# Visualize C only over land so zeros don't flatten the ramp
m.addLayer(C.updateMask(land_mask), {'min':0, 'max':1, 'palette':['#2c7bb6','#ffffbf','#d7191c']}, 'C (land only)')
m.addLayer(water_mask.selfMask(), {'palette':['#3b9ddd']}, 'Water (SCL+WC)')
m.addLayer(built_mask.selfMask(), {'palette':['#ff7f00']}, 'Built-up (WC+DW)')
m.centerObject(aoi, 11)
m

Mean C (land only): 0.20948183926929606
AOI area m^2: 402087834.16301686
Non-soil (water or built) area m^2: 12181964.545730919


Map(center=[31.52869697983162, -96.8361348309582], controls=(WidgetControl(options=['position', 'transparent_b…

##### Exporting C-factor raster to google drive

In [2]:
# Make sure ee.Initialize() ran earlier
task = ee.batch.Export.image.toDrive(
    image=C,                                # your C image
    description='C_from_NDVI_water_built_zero',
    folder='geeMart_exports',                   # Drive folder (optional)
    fileNamePrefix='C_from_NDVI_water_built_zero_2024',
    region=aoi,                             # ee.Geometry is fine
    scale=SCALE_M,                          # 10 m
    maxPixels=1e13
    # crs='EPSG:32614',                     # optional: force CRS; otherwise inherits from C
    # fileFormat='GeoTIFF'                  # default is GeoTIFF
)
task.start()

In [9]:
geemap.ee_export_image_to_drive(
    image=C,
    description='C_from_NDVI_water_built_zero',
    folder='gee_exports',
    fileNamePrefix='C_from_NDVI_water_built_zero_2024',
    region=aoi,
    scale=SCALE_M,
    # crs='EPSG:32614',
    maxPixels=1e13
)